# Multi-Agent Negotiation AI Showcase
### Reinforcement Learning using self-play (PPO)

This notebook demonstrates the underlying logic and performance of the negotiation agents trained in the `NegotiationEnv` environment.

## 1. Setup & Environment
We use a custom Gymnasium environment where agents negotiate over a price. The Buyer has a budget, and the Seller has a floor price.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

# local imports
from env.negotiation_env import NegotiationEnv

print("Setup complete.")

## 2. Load Trained Agents
We load the models from the `models/` directory.

In [ ]:
buyer_model = PPO.load("models/trained_buyer.zip")
seller_model = PPO.load("models/trained_seller.zip")
print("Models loaded successfully.")

## 3. Sample Negotiation Session
Let's simulate one single negotiation session and see how they haggle.

In [ ]:
env = NegotiationEnv(role='buyer', opponent_model=seller_model)
obs, info = env.reset(options={'buyer_budget': 120, 'seller_floor': 85})

done = False
while not done:
    action, _ = buyer_model.predict(obs, deterministic=False)
    obs, reward, done, truncated, info = env.step(action)

print("Negotiation Log:")
for msg in info['chat_log']:
    print(msg)

print(f"\nFinal Result: {'DEAL REACHED at $' + str(info['deal_price']) if info['deal_reached'] else 'NO DEAL'}")

## 4. Visualizing the Concession Curve
The 'Concession Curve' shows how both agents movement towards each other over time.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(info['buyer_history'], label='Buyer Offers', marker='o', color='blue')
plt.plot(info['seller_history'], label='Seller Offers', marker='o', color='green')
plt.axhline(y=info['buyer_budget'], color='blue', linestyle='--', alpha=0.3, label='Buyer Budget')
plt.axhline(y=info['seller_floor'], color='green', linestyle='--', alpha=0.3, label='Seller Floor')

plt.title("Negotiation Concession Curves")
plt.xlabel("Round")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Summary Statistics
In our self-play environment, agents learn to find the 'Zone of Possible Agreement' (ZOPA) organically.